In [1]:
from ERA_Distribution_Classes_Python.Classes.ERADist import ERADist
from ERA_Distribution_Classes_Python.Classes.ERANataf import ERANataf
from ERA_Distribution_Classes_Python.Classes.FORM_HLRF import FORM_HLRF
from ERA_Distribution_Classes_Python.Classes.FORM_fmincon import FORM_fmincon
from ERA_Distribution_Classes_Python.Classes.SuS import SuS

In [2]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt 
from structure import Structure
from solver_1st import Solver1stOrder
from measures_of_nonlinearity import kappa_1, kappa_2, kappa_12, r1, r2
from syst_4_model_functions import t_S_linear, t_S_hyperplane_linear

## Material Properties

In [3]:
# approximately resembles an IPE 120
E = 210e6 # kN/m2 
A = 1.321e-3 # m2
I = 0.318e-5 # m4
h = 0.12  # m
z = h/2  # m
alpha = 1.14

In [4]:
def t_R(M_k, I=I, z=z, alpha=alpha):
    """
    Takes in the Steel Bending Strength M_k (random variable) in kN/cm2
    I in m^4
    z in m
    alpha is the plastic ratio 

    Returns the characteristic Bending Moment Resistance M_c_Rk for the given system in kNm
    """
    return I/z * M_k * 100e2 * alpha

In [5]:
# Quick Check: should give M_Rd = 21.449 kNm
print(t_R(35.5))

21.449099999999994


## System Definition

In [6]:
# Vectorized Version of the Structural response function (Better for array handling later)
t_S_vectorized = np.vectorize(t_S_hyperplane_linear, otypes=[float])

## Target characteristic values for calibrating random variables

In [7]:
s_k = 1.1 # snow load on ground kN/m2
q_b = 0.65 # wind pressure kN/m2
w_k = q_b * 0.8 # wind load kN/m2 with c_pe,10 = 0.8 (Area D)
m_k = 30.93956533784639 # kN/cm2 (steel yield resistance) calibrated for eta = 100 % in linear case

## Random Variables

In [8]:
# Snow time-invariant part
mu_Theta_1 = 0.81
cov_Theta_1 = 0.26
sig_Theta_1 = mu_Theta_1 * cov_Theta_1
Theta_L1 = ERADist('lognormal','MOM',[mu_Theta_1, sig_Theta_1])

# Snow load on ground
mu_L1 = 1.0
cov_L1 = 0.2
sig_L1 = mu_L1 * cov_L1
L1 = ERADist('gumbel','MOM',[mu_L1,sig_L1])

In [9]:
percentile_L1 = L1.icdf(0.98)
print(f"Snow 98% Percentile: {percentile_L1}")

Snow 98% Percentile: 1.5184551765313796


In [10]:
# Wind time-invariant part
mu_Theta_2 = 0.97
cov_Theta_2 = 0.26
sig_Theta_2 = mu_Theta_2 * cov_Theta_2
Theta_L2 = ERADist('lognormal','MOM',[mu_Theta_2, sig_Theta_2])

# Wind velocity pressure
mu_L2 = 1.0 
cov_L2 = 0.14
sig_L2 = mu_L2 * cov_L2
L2 = ERADist('gumbel','MOM',[mu_L2, sig_L2])

In [11]:
percentile_L2 = L2.icdf(0.98)
print(f"Wind 98% Percentile: {percentile_L2}")

Wind 98% Percentile: 1.3629186235719657


In [12]:
# Structural Response Model Uncertainty (from JCSS Probabilistic Model Code, Part 3, Table 3.9.1)
mu_Theta_S = 1.0
cov_Theta_S = 0.1
sig_Theta_S = mu_Theta_S * cov_Theta_S
Theta_S = ERADist('lognormal','MOM',[mu_Theta_S, sig_Theta_S])  # Distribution for Moments in frames

In [13]:
# Steel bending model uncertainty
mu_Theta_M = 1.15
cov_Theta_M = 0.05
sig_Theta_M = mu_Theta_M * cov_Theta_M
Theta_M = ERADist('lognormal','MOM',[mu_Theta_M, sig_Theta_M])

# Steel yielding strength
mu_M = 1.0
cov_M = 0.05
sig_M = mu_M * cov_M
M = ERADist('lognormal','MOM',[mu_M, sig_M])

In [14]:
percentile_M = M.icdf(0.05)
print(f"Steel 5% Percentile: {percentile_M}")

Steel 5% Percentile: 0.9199464756612658


## Shifting / Scaling Random Variables 

In [15]:
# Snow Load on Ground, shifted to characteristic value
snow_shift = s_k / percentile_L1 # ratio of target to current percentile, by which mean and std get multiplied

mu_L1_shifted = mu_L1 * snow_shift
sig_L1_shifted = sig_L1 * snow_shift
L1_shifted = ERADist('gumbel','MOM',[mu_L1_shifted, sig_L1_shifted])

print(f"""Snow Load on Ground gets shifted by {snow_shift}""")
print(f"""Old mean: {mu_L1}; New mean: {mu_L1_shifted}""")
print(f"""Old std: {sig_L1}; New std: {sig_L1_shifted}""")
print(f"""Old 98th percentile: {L1.icdf(.98)}; New 98th percentile: {L1_shifted.icdf(.98)}""")
print(f"""Old COV: {L1.std()/L1.mean()}; New COV: {L1_shifted.std()/L1_shifted.mean()}""")

Snow Load on Ground gets shifted by 0.7244204616646898
Old mean: 1.0; New mean: 0.7244204616646898
Old std: 0.2; New std: 0.14488409233293795
Old 98th percentile: 1.5184551765313796; New 98th percentile: 1.0999999999999999
Old COV: 0.19999999999999998; New COV: 0.19999999999999996


In [16]:
# Wind velocity pressure, shifted to characteristic value
wind_shift = q_b / percentile_L2

mu_L2_shifted = mu_L2 * wind_shift
sig_L2_shifted = sig_L2 * wind_shift
L2_shifted = ERADist('gumbel','MOM',[mu_L2_shifted, sig_L2_shifted])

print(f"""Wind velocity pressure gets shifted by {wind_shift}""")
print(f"""Old mean: {mu_L2}; New mean: {mu_L2_shifted}""")
print(f"""Old std: {sig_L2}; New std: {sig_L2_shifted}""")
print(f"""Old 98th percentile: {L2.icdf(.98)}; New 98th percentile: {L2_shifted.icdf(.98)}""")
print(f"""Old COV: {L2.std()/L2.mean()}; New COV: {L2_shifted.std()/L2_shifted.mean()}""")

Wind velocity pressure gets shifted by 0.4769176888173018
Old mean: 1.0; New mean: 0.4769176888173018
Old std: 0.14; New std: 0.06676847643442226
Old 98th percentile: 1.3629186235719657; New 98th percentile: 0.65
Old COV: 0.14; New COV: 0.13999999999999999


In [17]:
# Steel bending resistance, shifted to characteristic value
steel_shift = m_k / percentile_M

mu_M_shifted = mu_M * steel_shift
sig_M_shifted = sig_M * steel_shift
M_shifted = ERADist('lognormal','MOM',[mu_M_shifted, sig_M_shifted])

print(f"""Steel bending resistance gets shifted by {steel_shift}""")
print(f"""Old mean: {mu_M}; New mean: {mu_M_shifted}""")
print(f"""Old std: {sig_M}; New std: {sig_M_shifted}""")
print(f"""Old 5th percentile: {M.icdf(0.05)}; New 5th percentile: {M_shifted.icdf(0.05)}""")
print(f"""Old COV: {M.std()/M.mean()}; New COV: {M_shifted.std()/M_shifted.mean()}""")

Steel bending resistance gets shifted by 33.63191898268511
Old mean: 1.0; New mean: 33.63191898268511
Old std: 0.05; New std: 1.6815959491342554
Old 5th percentile: 0.9199464756612658; New 5th percentile: 30.939565337846386
Old COV: 0.04999999999999947; New COV: 0.04999999999999947


## Characteristic values derived from Random Variables for further calculation

In [18]:
l_1k = L1_shifted.icdf(0.98)
l_2k = L2_shifted.icdf(0.98)
m_k = M_shifted.icdf(0.05)
print(f"l_1k = {l_1k:.4f} kN/m^2")
print(f"l_2k = {l_2k:.4f} kN/m^2")
print(f"m_k = {m_k:.4f} kN/cm^2")

l_1k = 1.1000 kN/m^2
l_2k = 0.6500 kN/m^2
m_k = 30.9396 kN/cm^2


## Partial Safety Factors

In [19]:
gamma_M = 1.0  # Resistance
gamma_F1 = 1.5   # Snow Load
gamma_F2 = 1.5   # Wind Load
#psi_0 = 1.0 # 0.6     # Windload

## Design Values

In [20]:
l_1d = s_k * gamma_F1
l_2d = q_b * gamma_F2 
m_d = m_k / gamma_M
print(f"l_1d = {l_1d:.4f} kN/m^2")
print(f"l_2d = {l_2d:.4f} kN/m^2")
print(f"m_d = {m_d:.4f} kN/cm^2")

l_1d = 1.6500 kN/m^2
l_2d = 0.9750 kN/m^2
m_d = 30.9396 kN/cm^2


## Measures of Nonlinearity

In [21]:
## Measures of Nonlinearity
k1 = kappa_1(l_1k=l_1k, l_1d=l_1d, t_S=t_S_hyperplane_linear)
k2 = kappa_2(l_2k=l_2k, l_2d=l_2d, t_S=t_S_hyperplane_linear)
k12 = kappa_12(l_1k=l_1k, l_1d=l_1d, l_2k=l_2k, l_2d=l_2d, t_S=t_S_hyperplane_linear)
r1 = r1(l_1k=l_1k, l_2k=l_2k, t_S=t_S_hyperplane_linear)
r2 = r2(l_1k=l_1k, l_2k=l_2k, t_S=t_S_hyperplane_linear)


print(f"kappa1 = {k1}")
print(f"kappa2 = {k2}")
print(f"kappa12 = {k12}")
print(f"r1 = {r1}")
print(f"r2 = {r2}")

kappa1 = 1.0000000000000004
kappa2 = 1.0
kappa12 = 0.9999999999999998
r1 = 0.11572415990296812
r2 = 0.8842758400970319


## Design parameters p for option 1 and 2 

In [22]:
# Design Opt 1
e_d_1 = t_S_hyperplane_linear(l_1d, l_2d)

# Design Opt 2
argument_1 = gamma_F1 * t_S_hyperplane_linear(l_1k,  (gamma_F2 / gamma_F1) * l_2k)
argument_2 = gamma_F2 * t_S_hyperplane_linear((gamma_F1 / gamma_F2) * l_1k, l_2k)
e_d_2 = max(argument_1, argument_2)

p_opt1 = gamma_M * e_d_1 / t_R(M_k=m_d)
p_opt2 = gamma_M * e_d_2 / t_R(M_k=m_d)

print("Design Option 1:")
print(f"e_d = {e_d_1} MPa")
print(f"p_opt1 = {p_opt1}")
print(f"\n")
print("Design Option 2:")
print(f"e_d = {e_d_2} MPa")
print(f"p_opt2 = {p_opt2}")

Design Option 1:
e_d = 18.693685377126783 MPa
p_opt1 = 1.0


Design Option 2:
e_d = 18.693685377126783 MPa
p_opt2 = 1.0


## Construction of the Nataf Distribution

In [23]:
# Array of marginal distributions
marginal_dist = [Theta_M, M_shifted, Theta_L1, L1_shifted, Theta_L2, L2_shifted, Theta_S]

# Correlation matrix (no correlation yet)
dimensions = len(marginal_dist)
R_xx = np.eye(dimensions)

# Construction of the Nataf Distribution
nataf = ERANataf(M=marginal_dist, Correlation=R_xx)

## Subset Simulation

### Design Option 1

In [24]:
# deterministic design action effect
e_d_opt1 = t_S_hyperplane_linear(l_1=gamma_F1 * s_k, l_2=gamma_F2 * q_b) # kNm

In [25]:
def g_opt_1_sus(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt1 / r_k) * x[:,0] * t_R(M_k=x[:,1])
    action_side = x[:,6] * t_S_vectorized(l_1=(x[:,2] * x[:,3]), l_2=(x[:,4] * x[:,5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [26]:
# # %% Samples Return
# samples_return = 1
# # %% subset simulation
# N  = 10000        # Total number of samples for each level
# p0 = 0.1         # Probability of each subset, chosen adaptively

# print('\n\nSUBSET SIMULATION: ')
# [Pf_SuS_1, delta_SuS, b, Pf, b_sus, pf_sus, samplesU, samplesX, fs_iid] = SuS(N, p0, g_opt_1_sus, nataf, samples_return)

In [27]:
# print("Subset Simulation for Design Option 1")
# print(f"P(F) = {Pf_SuS_1}")
# X = sp.stats.Normal()
# beta = - X.icdf(Pf_SuS_1)
# print(f"beta = {beta}")
# print(samplesX)

---

### Design Option 2

In [28]:
# deterministic design action effect
argument_1 = gamma_F1 * t_S_hyperplane_linear(l_1= s_k, l_2= (gamma_F2 / gamma_F1) * q_b)
argument_2 = gamma_F2 * t_S_hyperplane_linear(l_1= (gamma_F1 / gamma_F2) * s_k, l_2= q_b)
e_d_opt2 = max(argument_1, argument_2) # kNm

In [29]:
def g_opt_2_sus(x):
    """
    LSF for Design Option 2
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt2 / r_k) * x[:,0] * t_R(M_k=x[:,1])
    action_side = x[:,6] * t_S_vectorized(l_1=(x[:,2] * x[:,3]), l_2=(x[:,4] * x[:,5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [30]:
# # %% Samples Return
# samples_return = 1
# # %% subset simulation
# N  = 10000        # Total number of samples for each level
# p0 = 0.1         # Probability of each subset, chosen adaptively

# print('\n\nSUBSET SIMULATION: ')
# [Pf_SuS_2, delta_SuS, b, Pf, b_sus, pf_sus, samplesU, samplesX, fs_iid] = SuS(N, p0, g_opt_2_sus, nataf, samples_return)

In [31]:
# print("Subset Simulation for Design Option 2")
# print(f"P(F) = {Pf_SuS_2}")

# X = sp.stats.Normal()
# beta = - X.icdf(Pf_SuS_2)
# print(f"beta = {beta}")
# # print(samplesX)

## FORM Analysis

### Design Option 1

In [32]:
def g_opt_1_FORM(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt1 / r_k) * x[0] * t_R(M_k=x[1])
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [33]:
# Perform FORM with HLRF
# [u_star, x_star, beta, Pf, S_F1, S_F1_T] = FORM_HLRF(g=g_opt_1, dg=[], distr=nataf, sensitivity_analysis=0, u0=0)

# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_1_FORM, dg=[], distr=nataf)


*scipy.optimize.minimize() with  SLSQP  Method

  11  iterations... Reliability index =  3.4310315917223  --- Failure probability =  0.00030064528535868546 




In [34]:
print(f"u_star = {u_star}")
print(f"x_star = {x_star}")
print(f"(alpha_2)^2 = {(u_star/beta)**2}")
print(f"beta = {beta}")
print(f"P(F) = {Pf}")
print(f"g(X*) = {g_opt_1_FORM(x_star)}")

u_star = [-0.51733257 -0.51733261  0.09882933  0.07259142  2.55033684  1.91045663
  1.03325335]
x_star = [ 1.1192547  32.73276815  0.80400401  0.71018015  1.8023803   0.63220089
  1.10306564]
(alpha_2)^2 = [2.27347513e-02 2.27347545e-02 8.29702193e-04 4.47632029e-04
 5.52517013e-01 3.10045143e-01 9.06910035e-02]
beta = 3.4310315917223
P(F) = 0.00030064528535868546
g(X*) = -2.9361629216850815e-07


### Design Option 2

In [35]:
def g_opt_2_FORM(x):
    """
    LSF for Design Option 2
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt2 / r_k) * x[0] * t_R(M_k=x[1])
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [36]:
# Perform FORM with HLRF
# [u_star, x_star, beta, Pf, S_F1, S_F1_T] = FORM_HLRF(g=g_opt_2, dg=[], distr=nataf, sensitivity_analysis=0, u0=1)

# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_2_FORM, dg=[], distr=nataf)


*scipy.optimize.minimize() with  SLSQP  Method

  11  iterations... Reliability index =  3.4310315917223  --- Failure probability =  0.00030064528535868546 




In [37]:
print(f"u_star = {u_star}")
print(f"x_star = {x_star}")
print(f"(alpha_2)^2 = {(u_star/beta)**2}")
print(f"beta = {beta}")
print(f"P(F) = {Pf}")
print(f"g(X*) = {g_opt_2_FORM(x_star)}")

u_star = [-0.51733257 -0.51733261  0.09882933  0.07259142  2.55033684  1.91045663
  1.03325335]
x_star = [ 1.1192547  32.73276815  0.80400401  0.71018015  1.8023803   0.63220089
  1.10306564]
(alpha_2)^2 = [2.27347513e-02 2.27347545e-02 8.29702193e-04 4.47632029e-04
 5.52517013e-01 3.10045143e-01 9.06910035e-02]
beta = 3.4310315917223
P(F) = 0.00030064528535868546
g(X*) = -2.9361629216850815e-07
